In [1]:
# import
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
from scipy import stats

In [ ]:
# load data
df = pd.read_csv("../data/test_data.csv")

# rename relevant columns
df = df.rename(columns = {"Participant Code": "subject",
                          "sender" : "screen_type",
                          "duration" : "screen_duration",
                          "sent" : "sequence",
                          "cond": "condition",
                          "type" : "error_type",
                          "response" : "response",
                          "correctResponse": "correct_response",
                          "correct": "accuracy"})

# filter to relevant columns
df = df[["subject", "screen_type", "screen_duration", "sequence", "condition", 
         "error_type", "response", "correct_response", "accuracy"]]

# filter out instr screens, practice rounds, etc by keeping only rows with a sequence
df = df.dropna(subset = ["sequence"])

# check
df.head(15)

,Participant Code,Stable ID,Group Code,Task Group Code,Group Code Match,custom-fbclid,Submitted At,Task,Task Version,Task File Revision,...,sent,time_commit,time_end,time_render,time_run,time_show,time_switch,timestamp,type,url
0,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,13588.1,13587.0,8358.146,8372.4,8374.8,13642.1,2026-11-10T11:02:00.230Z,NaN,{}
1,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,14787.1,14783.0,13592.024,13588.5,13642.1,14841.7,2026-11-10T11:02:01.429Z,NaN,NaN
2,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,15160.6,15158.8,14775.190,14787.5,14841.7,15176.1,2026-11-10T11:02:01.803Z,NaN,NaN
3,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,15660.2,15658.8,15158.800,15160.8,15176.1,15674.8,2026-11-10T11:02:02.302Z,NaN,NaN
4,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,15836.2,15834.4,15658.800,15661.8,15674.8,15841.6,2026-11-10T11:02:02.478Z,NaN,NaN
5,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,16027.5,16025.4,15824.800,15837.5,15841.6,16041.6,2026-11-10T11:02:02.670Z,NaN,NaN
6,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,16027.9,16025.4,14775.190,14787.4,14841.7,16041.6,2026-11-10T11:02:02.670Z,NaN,NaN
7,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,16343.7,16342.2,16025.400,16028.1,16041.6,16358.1,2026-11-10T11:02:02.986Z,NaN,NaN
8,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,16493.6,16492.3,16342.200,16343.9,16358.1,16508.4,2026-11-10T11:02:03.136Z,NaN,NaN
9,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,16509.3,16507.1,16492.300,16494.5,NaN,16524.9,2026-11-10T11:02:03.151Z,NaN,NaN


In [ ]:
# define function for compressing data to one row per trial
def compress_data(trial):
    # get presentation duration as logged on stimuli screen
    pres_dur = trial.loc[trial["screen_type"] == "stimuli", 
                         "screen_duration"].values[0]

    # round presentation duration to nearest 50 (raw data is off due to screen refresh rate)
    pres_dur = round(pres_dur / 50) * 50
    
    # calculate RT as sum of raw presentation & mask duration
    rt = trial.loc[trial["screen_type"].isin(["stimuli", "mask"]), 
                   "screen_duration"].sum()
    
    # get accuracy, response, and correct response as logged on mask screen
    acc = trial.loc[trial["screen_type"] == "mask", 
                    "accuracy"].values[0]
    res = trial.loc[trial["screen_type"] == "mask", 
                    "response"].values[0]
    cor_res = trial.loc[trial["screen_type"] == "mask", 
                        "correct_response"].values[0]
    
    # get grammaticality condition and error type (invariant for the trial)
    cond = trial["condition"].values[0]
    error_type = trial["error_type"].values[0]
    
    return pd.Series({"presentation_duration": pres_dur,
                      "RT": rt,
                      "condition": cond,
                      "error_type": error_type,
                      "response": res,
                      "correct_response" : cor_res,
                      "accuracy": acc})

In [5]:
# apply compression function at trial level as defined by subject & sentence
df_compressed = df.groupby(["subject", "sequence"]).apply(compress_data).reset_index()

# check
df_compressed.head(15)

,subject,sequence,presentation_duration,RT,condition,error_type,response,correct_response,accuracy
0,58MSSZ47,Bilen er stor og grim,100,113.3,g,2.0,NaN,NaN,NaN
1,58MSSZ47,Bilen ramte den lille pige,150,583.7,g,2.0,u,g,False
2,58MSSZ47,De driftig vand til maden,0,4.3,u,4.0,NaN,NaN,NaN
3,58MSSZ47,De drikker kaffe på altanen,150,216.9,g,3.0,u,g,False
4,58MSSZ47,De en fik ny hund,200,296.3,u,5.0,u,u,True
5,58MSSZ47,De gik en lang tur,250,272.5,g,5.0,u,g,False
6,58MSSZ47,De kan måske lide ham,200,187.8,g,5.0,NaN,NaN,NaN
7,58MSSZ47,De skal handle i Tyskland,150,136.3,g,3.0,NaN,NaN,NaN
8,58MSSZ47,De spisede mad i går,250,310.3,u,3.0,u,u,True
9,58MSSZ47,De vil ikke spiser mad,0,11.7,u,3.0,NaN,NaN,NaN


In [6]:
# define function for excluding participants and sequences with low accuracy
def filter_accuracy(df, threshold = 0.5):
    # participants
    subject_acc = df.groupby("subject")["accuracy"].mean()
    valid_subjects = subject_acc[subject_acc >= threshold].index
    df = df[df["subject"].isin(valid_subjects)]

    # sequences
    sequence_acc = df.groupby("sequence")["accuracy"].mean()
    valid_sequences = sequence_acc[sequence_acc >= threshold].index
    df = df[df["sequence"].isin(valid_sequences)]
   
    return df

In [7]:
# apply filter function
df_clean = filter_accuracy(df_compressed)

# check
df_clean.head(15)

,subject,sequence,presentation_duration,RT,condition,error_type,response,correct_response,accuracy
0,58MSSZ47,Bilen er stor og grim,100,113.3,g,2.0,NaN,NaN,NaN
1,58MSSZ47,Bilen ramte den lille pige,150,583.7,g,2.0,u,g,False
4,58MSSZ47,De en fik ny hund,200,296.3,u,5.0,u,u,True
7,58MSSZ47,De skal handle i Tyskland,150,136.3,g,3.0,NaN,NaN,NaN
8,58MSSZ47,De spisede mad i går,250,310.3,u,3.0,u,u,True
9,58MSSZ47,De vil ikke spiser mad,0,11.7,u,3.0,NaN,NaN,NaN
10,58MSSZ47,Den lille fugl var sød,100,241.1,g,1.0,u,g,False
12,58MSSZ47,Det er et fint frugt,200,180.0,u,1.0,NaN,NaN,NaN
13,58MSSZ47,Det er et gult bil,350,356.1,u,1.0,u,u,True
14,58MSSZ47,Det er et stort hus,100,111.0,g,1.0,g,g,True


In [ ]:
# set up confusion matrix for possible outcomes
matrix = [(df_clean["condition"] == "g") & (df_clean["accuracy"] == True),   # hit
          (df_clean["condition"] == "g") & (df_clean["accuracy"] == False),  # miss
          (df_clean["condition"] == "u") & (df_clean["accuracy"] == True),   # correct rejection
          (df_clean["condition"] == "u") & (df_clean["accuracy"] == False)]  # false alarm

# define outcome labels
labels = ["hit", "miss", "CR", "FA"]

# select label for each row and add to new columnp
df_clean["outcome"] = np.select(matrix, labels, default = "unknown")

# check
df_clean.head(15)

,subject,sequence,presentation_duration,RT,condition,error_type,response,correct_response,accuracy,outcome
0,58MSSZ47,Bilen er stor og grim,100,113.3,g,2.0,NaN,NaN,NaN,unknown
1,58MSSZ47,Bilen ramte den lille pige,150,583.7,g,2.0,u,g,False,miss
4,58MSSZ47,De en fik ny hund,200,296.3,u,5.0,u,u,True,CR
7,58MSSZ47,De skal handle i Tyskland,150,136.3,g,3.0,NaN,NaN,NaN,unknown
8,58MSSZ47,De spisede mad i går,250,310.3,u,3.0,u,u,True,CR
9,58MSSZ47,De vil ikke spiser mad,0,11.7,u,3.0,NaN,NaN,NaN,unknown
10,58MSSZ47,Den lille fugl var sød,100,241.1,g,1.0,u,g,False,miss
12,58MSSZ47,Det er et fint frugt,200,180.0,u,1.0,NaN,NaN,NaN,unknown
13,58MSSZ47,Det er et gult bil,350,356.1,u,1.0,u,u,True,CR
14,58MSSZ47,Det er et stort hus,100,111.0,g,1.0,g,g,True,hit


In [ ]:
# group data by participant and presentation duration and find hit rate and false alarm rate; force 1.0 to 0.99 and 0.0 to 0.01

In [ ]:
# calculate sensitivity and bias for each participant at each presentation duration

In [ ]:
# get mean sensitivity and bias at each presentation duration across participants

In [ ]:
# do some stuff with RT

In [ ]:
# plotting!!!